# Inference Sentiment Analysis

Demo prediksi sentimen dari input teks. Mendukung: SVM/TF-IDF (EXP-02), Ensemble (EXP-04), IndoBERT (EXP-05), IndoBERT Tuned (EXP-06).

In [ ]:
import os, joblib, re
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
SENTIMENT_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
SENTIMENT_LABELS = ['Negative', 'Neutral', 'Positive']

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## Pilih Model (Ubah `MODEL_DIR` sesuai keinginan)

In [ ]:
# === PILIH MODEL DI SINI ===
# Opsi: EXP-01_LR_TFIDF, EXP-02_SVM_TFIDF, EXP-03_LR_BoW, EXP-04_Ensemble
MODEL_DIR = '../models/EXP-02_SVM_TFIDF'
print(f'Selected model: {MODEL_DIR}')

In [ ]:
# Load TF-IDF / BoW / Ensemble model
def load_sklearn_model(model_dir):
    model = joblib.load(f'{model_dir}/model.pkl')
    vectorizer = joblib.load(f'{model_dir}/vectorizer.pkl')
    return model, vectorizer

def predict_sklearn(text, model, vectorizer):
    clean = clean_text(text)
    vec = vectorizer.transform([clean])

    # Ensemble model is dict of {'lr': ..., 'svm': ...}
    if isinstance(model, dict):
        lr_proba = model['lr'].predict_proba(vec)
        svm_proba = model['svm'].predict_proba(vec)
        ensemble_proba = (lr_proba + svm_proba) / 2
        pred = np.argmax(ensemble_proba, axis=1)[0]
    elif hasattr(model, 'predict_proba'):
        pred = model.predict(vec)[0]
    else:
        pred = model.predict(vec)[0]
    return SENTIMENT_MAP[pred]

model, vectorizer = load_sklearn_model(MODEL_DIR)
print(f'Loaded model from {MODEL_DIR}')

## Demo Inference: TF-IDF / BoW / Ensemble Models

In [ ]:
# Contoh 1: Ulasan positif
sample1 = "Aplikasi sangat membantu transaksi sehari-hari, fitur lengkap dan mudah digunakan"
result1 = predict_sklearn(sample1, model, vectorizer)
print(f'Input: {sample1}')
print(f'Sentimen: {result1}\n')

In [ ]:
# Contoh 2: Ulasan negatif
sample2 = "Aplikasi sering error dan lambat, tidak bisa melakukan transaksi"
result2 = predict_sklearn(sample2, model, vectorizer)
print(f'Input: {sample2}')
print(f'Sentimen: {result2}\n')

In [ ]:
# Contoh 3: Ulasan netral
sample3 = "aplikasi standar, kadang lancar kadang lemot"
result3 = predict_sklearn(sample3, model, vectorizer)
print(f'Input: {sample3}')
print(f'Sentimen: {result3}\n')

In [ ]:
# Contoh 4: Ulasan dari dataset
df = pd.read_csv('../data/raw/reviews.csv')
for i in range(5):
    text = df['content'].iloc[i]
    score = df['score'].iloc[i]
    sent = predict_sklearn(text, model, vectorizer)
    print(f'Rating: {score} -> Predicted: {sent}')
    print(f'Text: {text[:80]}...\n')

## IndoBERT Inference (Ubah CELL DI BAWAH untuk memilih model)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# === PILIH MODEL INDOBERT ===
# Opsi: EXP-05_IndoBERT, EXP-06_IndoBERT_Tuned
INDODBERT_DIR = '../models/EXP-06_IndoBERT_Tuned'

tokenizer_indobert = AutoTokenizer.from_pretrained(INDODBERT_DIR)
model_indobert = AutoModelForSequenceClassification.from_pretrained(INDODBERT_DIR)
model_indobert.eval()
print(f'Loaded IndoBERT model from {INDODBERT_DIR}')

In [ ]:
def predict_indobert(text, model=model_indobert, tokenizer=tokenizer_indobert):
    clean = clean_text(text)
    inputs = tokenizer(clean, return_tensors='pt', truncation=True, padding=True, max_length=96)
    with torch.no_grad():
        outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=-1).item()
    return SENTIMENT_LABELS[pred]

# Test
test_samples = [
    "Aplikasi sangat membantu",
    "Aplikasi sering error",
    "aplikasi standar saja",
]
for s in test_samples:
    print(f'Input: {s} -> Sentimen: {predict_indobert(s)}')

## Batch Prediction Comparison (Semua Model)

In [ ]:
# Bandingkan semua model yang tersedia
all_models = {
    'EXP-01_LR_TFIDF': '../models/EXP-01_LR_TFIDF',
    'EXP-02_SVM_TFIDF': '../models/EXP-02_SVM_TFIDF',
    'EXP-04_Ensemble': '../models/EXP-04_Ensemble',
}

samples = [
    'aplikasi ini sangat bagus dan membantu sekali',
    'aplikasi lemot dan sering crash',
    'lumayan lah kadang lancar kadang error',
]

print(f'{"Model":<25} {"Positif":<15} {"Negatif":<15} {"Netral":<15}')
print('-' * 70)

for name, path in all_models.items():
    if not os.path.exists(f'{path}/model.pkl'):
        continue
    m, v = load_sklearn_model(path)
    results = [predict_sklearn(s, m, v) for s in samples]
    print(f'{name:<25} {results[0]:<15} {results[1]:<15} {results[2]:<15}')

# IndoBERT jika ada
if os.path.exists(f'{INDODBERT_DIR}/pytorch_model.bin'):
    ib_results = [predict_indobert(s) for s in samples]
    print(f'{"IndoBERT (" + INDODBERT_DIR.split("/")[-1] + ")":<25} {ib_results[0]:<15} {ib_results[1]:<15} {ib_results[2]:<15}')